In [ ]:
import time
import matplotlib.pyplot as plt

from experiment_helpers import (
    TASKS, setup, load_data, load_state, save_state,
    clear_model_results, record
)
from decision_tree import ScratchDecisionTree
from utils import prep, accuracy


MAX_DEPTHS = (2, 4, 6, 10, 15, None)
MIN_SAMPLES_SPLITS = (2, 5, 10, 20)
SCALINGS = ("none", "standardize")


def run_decision_tree(data, state):
    best = state["best"]
    clear_model_results(state, "decision_tree")
    curves = {}

    for task in TASKS:
        dataset = data[task]
        print(f"\n=== {task} ===")

        # Subsample features on image tasks to keep pure-Python tree loops fast
        max_features = "sqrt" if task == "image" else None
        resolved_max_features = (
            int(dataset["train_features"].shape[1] ** 0.5)
            if max_features == "sqrt"
            else None
        )

        for scaling in SCALINGS:
            train_x, val_x = prep(
                {"scaling": scaling},
                dataset["train_features"],
                dataset["val_features"]
            )

            for min_split in MIN_SAMPLES_SPLITS:
                scores = []

                for max_depth in MAX_DEPTHS:
                    model = ScratchDecisionTree(
                        max_depth=max_depth,
                        min_samples_split=min_split,
                        max_features=resolved_max_features,
                        seed=42
                    )
                    model.fit(train_x, dataset["train_labels"])
                    predictions = model.predict(val_x)

                    acc = accuracy(dataset["val_labels"], predictions)
                    scores.append(acc)

                    params = {
                        "max_depth": max_depth,
                        "min_samples_split": min_split,
                        "max_features": resolved_max_features,
                        "scaling": scaling,
                    }

                    record(best, task, "decision_tree", params, acc, predictions)

                curves[(task, scaling, min_split)] = scores

                depth_str = "  ".join(
                    f"d={d}: {a:.3f}"
                    for d, a in zip(MAX_DEPTHS, scores)
                )
                print(f"{scaling} min_split={min_split}: {depth_str}")

        print("Best Decision Tree:", best[task]["decision_tree"]["params"])
        print("Accuracy:", best[task]["decision_tree"]["acc"])

    x_labels = [str(d) if d is not None else "None" for d in MAX_DEPTHS]
    x_pos = range(len(MAX_DEPTHS))

    fig, graphs = plt.subplots(1, len(TASKS), figsize=(5 * len(TASKS), 3.5))
    if len(TASKS) == 1:
        graphs = [graphs]

    for graph, task in zip(graphs, TASKS):
        for min_split in MIN_SAMPLES_SPLITS:
            graph.plot(
                x_pos,
                curves[(task, "none", min_split)],
                marker="o",
                label=f"min_split={min_split}"
            )

        graph.set_title(f"Scratch Decision Tree: {task}")
        graph.set_xticks(list(x_pos))
        graph.set_xticklabels(x_labels)
        graph.set_xlabel("max_depth")
        graph.set_ylabel("validation accuracy")
        graph.legend()

    plt.tight_layout()
    plt.savefig("plots/dt_depth.png", dpi=120)
    plt.close()


if __name__ == "__main__":
    setup()
    data = load_data()
    state = load_state()

    start = time.time()
    run_decision_tree(data, state)

    print(f"\nDecision Tree finished in {time.time() - start:.1f}s")
    save_state(state)